# MTurk Example Selector

This notebook filters the full Task A / Task B MTurk candidate CSVs down to balanced 100-row MTurk exports.

- **Task A**: select **50 pairs** total (100 rows), with at least 10 pairs from each model and a final 13/13/12/12-style split.
- **Task B**: select **25 rows from each of the 4 models** (100 rows total).
- LLM judgments are cached, and the notebook evaluates **model-by-model**, so reruns do not query GPT again for models that already have enough passing examples.


In [ ]:
import os
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path('/playpen-ssd/smerrill/deception2')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from mturk_dataset_utils import (
    MODEL_VARIANTS,
    MTURK_OUTPUT_ROOT,
    allocate_counts,
    evaluate_taska_with_llm,
    evaluate_taskb_with_llm,
    select_taska_subset,
    select_taskb_subset,
    selector_summary_dataframe,
    write_json,
)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


In [ ]:
# Configuration
TASKA_INPUT_PATH = MTURK_OUTPUT_ROOT / 'taska.csv'
TASKB_INPUT_PATH = MTURK_OUTPUT_ROOT / 'taskb.csv'
SELECTOR_OUTPUT_ROOT = MTURK_OUTPUT_ROOT / 'selector_outputs'
SELECTOR_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

OPENAI_MODEL_NAME = os.environ.get('MTURK_SELECTOR_MODEL', 'gpt-4o-mini')
OPENAI_API_MODE = os.environ.get('MTURK_SELECTOR_API_MODE', 'auto')
OPENAI_BASE_URL = os.environ.get('MTURK_SELECTOR_BASE_URL') or None

MODEL_IDS = list(MODEL_VARIANTS.keys())
MODEL_ORDER = {model_id: idx for idx, model_id in enumerate(MODEL_IDS)}

TASKA_TARGET_PAIRS = 50
TASKA_TARGET_ROWS = TASKA_TARGET_PAIRS * 2
TASKA_MIN_PAIRS_PER_MODEL = 10
if TASKA_TARGET_PAIRS < TASKA_MIN_PAIRS_PER_MODEL * len(MODEL_IDS):
    raise ValueError('Task A target pairs must allow at least the requested minimum per model.')
TASKA_MODEL_PAIR_TARGETS = {model_id: TASKA_MIN_PAIRS_PER_MODEL for model_id in MODEL_IDS}
for model_id, extra in allocate_counts(
    TASKA_TARGET_PAIRS - TASKA_MIN_PAIRS_PER_MODEL * len(MODEL_IDS),
    MODEL_IDS,
).items():
    TASKA_MODEL_PAIR_TARGETS[model_id] += extra

TASKB_TARGET_ROWS = 100
if TASKB_TARGET_ROWS % len(MODEL_IDS) != 0:
    raise ValueError('Task B target rows must divide evenly across models.')
TASKB_MODEL_ROW_TARGETS = {model_id: TASKB_TARGET_ROWS // len(MODEL_IDS) for model_id in MODEL_IDS}

OVERWRITE_LLM_CACHE = False
TASKA_LLM_LIMIT = 0  # 0 means keep evaluating until each model reaches its Task A quota
TASKB_LLM_LIMIT = 0  # 0 means keep evaluating until each model reaches its Task B quota
SLEEP_SECONDS_BETWEEN_CALLS = 0.0

print('Configuration')
print(f'  Task A input: {TASKA_INPUT_PATH}')
print(f'  Task B input: {TASKB_INPUT_PATH}')
print(f'  Selector output root: {SELECTOR_OUTPUT_ROOT}')
print(f'  OpenAI model: {OPENAI_MODEL_NAME}')
print(f'  OpenAI api_mode: {OPENAI_API_MODE}')
print(f'  OpenAI base_url: {OPENAI_BASE_URL}')
print(f'  Task A target pairs: {TASKA_TARGET_PAIRS}')
print(f'  Task A target rows: {TASKA_TARGET_ROWS}')
print(f'  Task A per-model pair targets: {TASKA_MODEL_PAIR_TARGETS}')
print(f'  Task B target rows: {TASKB_TARGET_ROWS}')
print(f'  Task B per-model row targets: {TASKB_MODEL_ROW_TARGETS}')


In [ ]:
taska_df = pd.read_csv(TASKA_INPUT_PATH)
taskb_df = pd.read_csv(TASKB_INPUT_PATH)

print(f'Loaded Task A rows: {len(taska_df)}')
print(f'Loaded Task B rows: {len(taskb_df)}')
display(taska_df.head(3))
display(taskb_df.head(3))


In [ ]:
taska_model_judgments = []
taska_model_selected_rows = []
taska_model_selected_spikes = []
taska_model_target_rows = []

for model_idx, model_id in enumerate(MODEL_IDS):
    model_target_pairs = TASKA_MODEL_PAIR_TARGETS[model_id]
    model_taska_df = taska_df[taska_df['model_id'] == model_id].copy()
    model_judgments_df = evaluate_taska_with_llm(
        model_taska_df,
        output_root=SELECTOR_OUTPUT_ROOT,
        model_name=OPENAI_MODEL_NAME,
        base_url=OPENAI_BASE_URL,
        api_mode=OPENAI_API_MODE,
        overwrite=bool(OVERWRITE_LLM_CACHE and model_idx == 0),
        limit=TASKA_LLM_LIMIT,
        target_passing_examples=model_target_pairs,
        sleep_seconds=SLEEP_SECONDS_BETWEEN_CALLS,
    )
    model_passing_pairs = int(model_judgments_df['llm_action_correct'].fillna(False).astype(bool).sum())
    if model_passing_pairs < model_target_pairs:
        raise ValueError(
            f'Task A model {model_id} only has {model_passing_pairs} passing spike rows; '
            f'need {model_target_pairs}.'
        )

    model_selected_df, model_selected_spikes_df = select_taska_subset(
        model_taska_df,
        model_judgments_df,
        total_examples=model_target_pairs,
    )
    if len(model_selected_spikes_df) != model_target_pairs:
        raise ValueError(
            f'Task A model {model_id} selected {len(model_selected_spikes_df)} pairs; '
            f'expected {model_target_pairs}.'
        )

    taska_model_judgments.append(model_judgments_df)
    taska_model_selected_rows.append(model_selected_df)
    taska_model_selected_spikes.append(model_selected_spikes_df)
    taska_model_target_rows.append(
        {
            'model_id': model_id,
            'target_pairs': model_target_pairs,
            'passing_pairs': model_passing_pairs,
            'selected_pairs': int(model_selected_spikes_df['pair_id'].nunique()),
        }
    )
    print(
        f'Task A {model_id}: judged spike rows={len(model_judgments_df)} '
        f'passing_pairs={model_passing_pairs} selected_pairs={len(model_selected_spikes_df)} '
        f'target_pairs={model_target_pairs}'
    )

taska_judgments_df = pd.concat(taska_model_judgments, ignore_index=True)
taska_judgments_df = (
    taska_judgments_df.assign(model_order=taska_judgments_df['model_id'].map(MODEL_ORDER))
    .sort_values(['model_order', 'environment', 'pair_id'])
    .drop(columns=['model_order'])
    .reset_index(drop=True)
)

taska_selected_spikes_df = pd.concat(taska_model_selected_spikes, ignore_index=True)
taska_selected_spikes_df = (
    taska_selected_spikes_df.assign(model_order=taska_selected_spikes_df['model_id'].map(MODEL_ORDER))
    .sort_values(['model_order', 'environment', 'pair_id'])
    .drop(columns=['model_order'])
    .reset_index(drop=True)
)

taska_selected_df = pd.concat(taska_model_selected_rows, ignore_index=True)
role_order = {'pre_spike': 0, 'spike': 1}
taska_selected_df = (
    taska_selected_df.assign(
        model_order=taska_selected_df['model_id'].map(MODEL_ORDER),
        role_order=taska_selected_df['pair_role'].map(role_order),
    )
    .sort_values(['model_order', 'environment', 'pair_id', 'role_order', 'sentence_idx'])
    .drop(columns=['model_order', 'role_order'])
    .reset_index(drop=True)
)

taska_model_target_df = pd.DataFrame(taska_model_target_rows)
taska_model_target_df = (
    taska_model_target_df.assign(model_order=taska_model_target_df['model_id'].map(MODEL_ORDER))
    .sort_values(['model_order'])
    .drop(columns=['model_order'])
    .reset_index(drop=True)
)

taska_pass_summary_df = (
    taska_judgments_df.groupby(['model_id', 'environment'], dropna=False)
    .agg(
        n_spike_rows=('task_id', 'size'),
        n_correct=('llm_action_correct', 'sum'),
        mean_spike_delta=('spike_delta', 'mean'),
    )
    .reset_index()
)
taska_pass_summary_df['accuracy'] = taska_pass_summary_df['n_correct'] / taska_pass_summary_df['n_spike_rows']
taska_selected_summary_df = selector_summary_dataframe(taska_selected_df, task_name='taska')

print(f'Task A judged spike rows: {len(taska_judgments_df)}')
print(f'Task A selected pairs: {len(taska_selected_spikes_df)}')
print(f'Task A selected rows: {len(taska_selected_df)}')
display(taska_model_target_df)
display(taska_pass_summary_df)
display(taska_selected_summary_df)
display(taska_selected_df.head(10))


In [ ]:
taskb_model_judgments = []
taskb_model_selected_rows = []
taskb_model_target_rows = []

for model_idx, model_id in enumerate(MODEL_IDS):
    model_target_rows = TASKB_MODEL_ROW_TARGETS[model_id]
    model_taskb_df = taskb_df[taskb_df['model_id'] == model_id].copy()
    model_judgments_df = evaluate_taskb_with_llm(
        model_taskb_df,
        output_root=SELECTOR_OUTPUT_ROOT,
        model_name=OPENAI_MODEL_NAME,
        base_url=OPENAI_BASE_URL,
        api_mode=OPENAI_API_MODE,
        overwrite=bool(OVERWRITE_LLM_CACHE and model_idx == 0),
        limit=TASKB_LLM_LIMIT,
        target_passing_examples=model_target_rows,
        sleep_seconds=SLEEP_SECONDS_BETWEEN_CALLS,
    )
    model_passing_rows = int(model_judgments_df['llm_all_correct'].fillna(False).astype(bool).sum())
    if model_passing_rows < model_target_rows:
        raise ValueError(
            f'Task B model {model_id} only has {model_passing_rows} passing rows; '
            f'need {model_target_rows}.'
        )

    model_selected_df = select_taskb_subset(
        model_taskb_df,
        model_judgments_df,
        total_rows=model_target_rows,
    )
    if len(model_selected_df) != model_target_rows:
        raise ValueError(
            f'Task B model {model_id} selected {len(model_selected_df)} rows; '
            f'expected {model_target_rows}.'
        )

    taskb_model_judgments.append(model_judgments_df)
    taskb_model_selected_rows.append(model_selected_df)
    taskb_model_target_rows.append(
        {
            'model_id': model_id,
            'target_rows': model_target_rows,
            'passing_rows': model_passing_rows,
            'selected_rows': int(len(model_selected_df)),
        }
    )
    print(
        f'Task B {model_id}: judged rows={len(model_judgments_df)} '
        f'passing_rows={model_passing_rows} selected_rows={len(model_selected_df)} '
        f'target_rows={model_target_rows}'
    )

taskb_judgments_df = pd.concat(taskb_model_judgments, ignore_index=True)
taskb_judgments_df = (
    taskb_judgments_df.assign(model_order=taskb_judgments_df['model_id'].map(MODEL_ORDER))
    .sort_values(['model_order', 'environment', 'task_id'])
    .drop(columns=['model_order'])
    .reset_index(drop=True)
)

taskb_selected_df = pd.concat(taskb_model_selected_rows, ignore_index=True)
taskb_selected_df = (
    taskb_selected_df.assign(model_order=taskb_selected_df['model_id'].map(MODEL_ORDER))
    .sort_values(['model_order', 'environment', 'task_id'])
    .drop(columns=['model_order'])
    .reset_index(drop=True)
)

taskb_model_target_df = pd.DataFrame(taskb_model_target_rows)
taskb_model_target_df = (
    taskb_model_target_df.assign(model_order=taskb_model_target_df['model_id'].map(MODEL_ORDER))
    .sort_values(['model_order'])
    .drop(columns=['model_order'])
    .reset_index(drop=True)
)

taskb_pass_summary_df = (
    taskb_judgments_df.groupby(['model_id', 'environment'], dropna=False)
    .agg(
        n_rows=('task_id', 'size'),
        n_action_correct=('llm_action_correct', 'sum'),
        n_commitment_correct=('llm_commitment_correct', 'sum'),
        n_all_correct=('llm_all_correct', 'sum'),
        mean_spike_delta=('spike_delta', 'mean'),
    )
    .reset_index()
)
taskb_pass_summary_df['all_correct_rate'] = taskb_pass_summary_df['n_all_correct'] / taskb_pass_summary_df['n_rows']
taskb_selected_summary_df = selector_summary_dataframe(taskb_selected_df, task_name='taskb')

print(f'Task B judged rows: {len(taskb_judgments_df)}')
print(f'Task B selected rows: {len(taskb_selected_df)}')
display(taskb_model_target_df)
display(taskb_pass_summary_df)
display(taskb_selected_summary_df)
display(taskb_selected_df.head(10))


In [ ]:
taska_judgments_path = SELECTOR_OUTPUT_ROOT / 'taska_gpt4o_mini_judgments.csv'
taska_selected_path = SELECTOR_OUTPUT_ROOT / 'taska_selected_100.csv'
taska_selected_spikes_path = SELECTOR_OUTPUT_ROOT / 'taska_selected_spike_rows.csv'
taska_selected_summary_path = SELECTOR_OUTPUT_ROOT / 'taska_selected_100_summary.csv'
taska_pass_summary_path = SELECTOR_OUTPUT_ROOT / 'taska_llm_pass_summary.csv'

taskb_judgments_path = SELECTOR_OUTPUT_ROOT / 'taskb_gpt4o_mini_judgments.csv'
taskb_selected_path = SELECTOR_OUTPUT_ROOT / 'taskb_selected_100.csv'
taskb_selected_summary_path = SELECTOR_OUTPUT_ROOT / 'taskb_selected_100_summary.csv'
taskb_pass_summary_path = SELECTOR_OUTPUT_ROOT / 'taskb_llm_pass_summary.csv'

taska_judgments_df.to_csv(taska_judgments_path, index=False)
taska_selected_df.to_csv(taska_selected_path, index=False)
taska_selected_spikes_df.to_csv(taska_selected_spikes_path, index=False)
taska_selected_summary_df.to_csv(taska_selected_summary_path, index=False)
taska_pass_summary_df.to_csv(taska_pass_summary_path, index=False)

taskb_judgments_df.to_csv(taskb_judgments_path, index=False)
taskb_selected_df.to_csv(taskb_selected_path, index=False)
taskb_selected_summary_df.to_csv(taskb_selected_summary_path, index=False)
taskb_pass_summary_df.to_csv(taskb_pass_summary_path, index=False)

write_json(
    SELECTOR_OUTPUT_ROOT / 'selector_run_config.json',
    {
        'taska_input_path': str(TASKA_INPUT_PATH),
        'taskb_input_path': str(TASKB_INPUT_PATH),
        'openai_model_name': OPENAI_MODEL_NAME,
        'openai_api_mode': OPENAI_API_MODE,
        'openai_base_url': OPENAI_BASE_URL,
        'taska_target_pairs': TASKA_TARGET_PAIRS,
        'taska_target_rows': TASKA_TARGET_ROWS,
        'taska_min_pairs_per_model': TASKA_MIN_PAIRS_PER_MODEL,
        'taska_model_pair_targets': TASKA_MODEL_PAIR_TARGETS,
        'taskb_target_rows': TASKB_TARGET_ROWS,
        'taskb_model_row_targets': TASKB_MODEL_ROW_TARGETS,
        'overwrite_llm_cache': OVERWRITE_LLM_CACHE,
        'taska_llm_limit': TASKA_LLM_LIMIT,
        'taskb_llm_limit': TASKB_LLM_LIMIT,
        'sleep_seconds_between_calls': SLEEP_SECONDS_BETWEEN_CALLS,
    },
)

print(f'Saved Task A judgments to:      {taska_judgments_path}')
print(f'Saved Task A selected rows to:  {taska_selected_path}')
print(f'Saved Task A spike rows to:     {taska_selected_spikes_path}')
print(f'Saved Task B judgments to:      {taskb_judgments_path}')
print(f'Saved Task B selected rows to:  {taskb_selected_path}')
print(f'Saved selector config to:       {SELECTOR_OUTPUT_ROOT / "selector_run_config.json"}')
